# Task A -- demojized MuRIL plus TF-IDF OOF ensemble

This notebook tests whether the current strong demojized MuRIL model and the demojized
TF-IDF baseline make complementary errors. It trains both models with the same
deduplicate-first five-fold split, then fits blend weights using only out-of-fold
predictions.

Models:
- demojized TF-IDF word and character n-grams with calibrated LinearSVC
- demojized MuRIL with the established six-epoch recipe

The notebook also performs a nested blend check. The raw OOF blend score is optimistic
because its weights are fitted on those rows; the nested score is the decision signal.
It creates validated ZIPs for both individual models and the ensemble. No full-data fit
is performed here.

Expected runtime is approximately 3--5 hours on a T4. Upload this notebook to Kaggle,
enable GPU and Internet, and choose Save Version -> Save & Run All.


In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Generate aligned five-fold predictions

Both commands use the repository's shared split seed 42 and the same deduplication policy.
The SVM runs on CPU while the MuRIL run uses the selected GPU. Their OOF matrices have
identical row order and shape, which is required for a valid blend.


In [ ]:
run([sys.executable, "-u", "-m", "hastika.models.baseline_svm",
     "--task", "a",
     "--tag", "task_a_svm_demojized",
     "--demojize"],
    log="artifacts/logs/task_a_svm_demojized.log")

run([sys.executable, "-u", "-m", "hastika.models.muril",
     "--tag", "task_a_muril_oof",
     "--folds", "5",
     "--seeds", "42",
     "--epochs", "6"],
    log="artifacts/logs/task_a_muril_oof.log")

for tag in ["task_a_svm_demojized", "task_a_muril_oof"]:
    d = pathlib.Path("artifacts/runs") / tag
    assert (d / "oof_probs.npy").exists(), f"missing OOF matrix for {tag}"
    assert (d / "test_probs.npy").exists(), f"missing validation probabilities for {tag}"
print("aligned OOF predictions are ready")


## 2. Search and evaluate the blend

The ensemble module checks individual OOF scores, searches the weight simplex, and runs
a nested five-fold weight fit. The nested estimate is the honest comparison against the
best single model. A blend is useful only if it survives that check.


In [ ]:
ENSEMBLE_TAGS = ["task_a_svm_demojized", "task_a_muril_oof"]
run([sys.executable, "-m", "hastika.models.ensemble",
     "--task", "a",
     "--tags", *ENSEMBLE_TAGS,
     "--samples", "20000",
     "--seed", "42",
     "--out", "task_a_svm_muril_ensemble"],
    log="artifacts/logs/task_a_svm_muril_ensemble.log")

assert (pathlib.Path("artifacts/runs/task_a_svm_muril_ensemble")
        / "predictions.csv").exists()
print("ensemble predictions are ready")


## 3. Validate all candidate submissions

All three files are checked against the Task A validation IDs and the required id,label
schema. The ensemble ZIP is a candidate; submit it only if the nested result supports
blending. Otherwise use the stronger individual ZIP.


In [ ]:
CANDIDATES = [
    ("task_a_svm_demojized", "task_a_svm_demojized"),
    ("task_a_muril_oof", "task_a_muril_oof"),
    ("task_a_svm_muril_ensemble", "task_a_svm_muril_ensemble"),
]
for tag, _ in CANDIDATES:
    pred = pathlib.Path("artifacts/runs") / tag / "predictions.csv"
    out = pathlib.Path("/kaggle/working") / f"{tag}.zip"
    run([sys.executable, "-m", "hastika.common.submission",
         "--task", "a", "--pred", str(pred), "--out", str(out)])
    with zipfile.ZipFile(out) as z:
        assert z.namelist() == ["predictions.csv"], z.namelist()
print("all three Task A ZIPs validated")


## 4. Preserve the outputs

Download this directory from Kaggle. It contains the individual and ensemble ZIPs,
OOF/test probability matrices, predictions, logs, and the nested blend report.


In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_ensemble_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for tag, _ in CANDIDATES:
    src = pathlib.Path("artifacts/runs") / tag
    shutil.copy2(pathlib.Path("/kaggle/working") / f"{tag}.zip",
                 OUT / f"{tag}.zip")
    for name in ["oof_probs.npy", "test_probs.npy", "predictions.csv"]:
        if (src / name).exists():
            shutil.copy2(src / name, OUT / f"{tag}_{name}")
for name in ["task_a_svm_demojized.log", "task_a_muril_oof.log",
             "task_a_svm_muril_ensemble.log"]:
    path = pathlib.Path("artifacts/logs") / name
    if path.exists():
        shutil.copy2(path, OUT / name)
print("download directory:", OUT)
print("preferred ensemble candidate:", OUT / "task_a_svm_muril_ensemble.zip")
